# 📊🤖 Mesa Financiera en Telegram — Ruta A (Google Colab)
**Dra. Elda C. Morales · Actividad para principiantes**

Construirás un ecosistema de **3 agentes** que trabajan en cadena, dirigidos por **Batuta**, el orquestador:

| Agente | Recibe | Hace | Entrega |
|---|---|---|---|
| 🛰️ **Iris · La Recolectora** (Agente 1) | Ticker que escribes en Telegram | Consulta datos reales de Alpha Vantage mediante **MCP** | Ficha JSON |
| 🧮 **Euler · El Analista** (Agente 2) | La ficha de Iris | Python **calcula** indicadores; OpenAI los **interpreta** | Análisis JSON |
| ✍️ **Vera · La Comunicadora** (Agente 3) | El análisis de Euler | Redacta una nota clara | Mensaje en Telegram |

**Cómo usar este notebook:** ejecuta las celdas en orden con ▶ (o `Shift + Enter`). Después de cada celda revisa el apartado *"Deberías ver"*.

> 💡 ¿Aún no tienes llaves? Deja `MODO_SIMULACION = True` en la celda 2: todo funciona con datos ficticios.

## Celda 1 · Instalar librerías
**Deberías ver:** la celda termina sin errores en rojo (puede tardar unos 20 segundos).

In [ ]:
!pip install -q openai==2.48.0 requests==2.32.5

## Celda 2 · Configuración y llaves
Las llaves **nunca** se escriben en el código. Guárdalas en el panel **🔑 Secretos** (barra izquierda de Colab) → *Add new secret*, con estos nombres exactos y el interruptor **Notebook access** encendido:

`TELEGRAM_BOT_TOKEN` · `TELEGRAM_CHAT_ID` · `OPEN_API_KEY` · `ALPHAVANTAGE_API_KEY`

**Deberías ver:** una tabla con ✅ en cada llave configurada y el modo (SIMULACIÓN o REAL).

In [ ]:
import json, math, os, re, statistics, time
import requests

MODO_SIMULACION = True        # ← cámbialo a False cuando tengas tus 4 llaves
MODO_CLASE      = True        # True = el bot muestra cada entrega entre agentes
MODELO          = "gpt-5.1"

def leer_secreto(nombre):
    """Lee un secreto de Colab (o de una variable de entorno fuera de Colab)."""
    try:
        from google.colab import userdata
        return (userdata.get(nombre) or "").strip()
    except Exception:
        return os.environ.get(nombre, "").strip()

TELEGRAM_TOKEN   = leer_secreto("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = leer_secreto("TELEGRAM_CHAT_ID")
OPEN_API_KEY     = leer_secreto("OPEN_API_KEY") or leer_secreto("OPENAI_API_KEY")
ALPHAVANTAGE_KEY = leer_secreto("ALPHAVANTAGE_API_KEY")

if not MODO_SIMULACION and not (OPEN_API_KEY and ALPHAVANTAGE_KEY):
    print("⚠️ Faltan OPEN_API_KEY o ALPHAVANTAGE_API_KEY: activo el MODO SIMULACIÓN.")
    MODO_SIMULACION = True

for nombre, valor in [("TELEGRAM_BOT_TOKEN", TELEGRAM_TOKEN), ("TELEGRAM_CHAT_ID", TELEGRAM_CHAT_ID),
                      ("OPEN_API_KEY", OPEN_API_KEY), ("ALPHAVANTAGE_API_KEY", ALPHAVANTAGE_KEY)]:
    print(f"{nombre:<22} {'✅ configurada' if valor else '⬜ vacía'}")   # nunca se imprime el valor
print("Modo:", "SIMULACIÓN (datos ficticios)" if MODO_SIMULACION else "REAL")

## Celda 3 · Conexión con OpenAI y con el servidor MCP
Aquí ocurre la integración **MCP**. El modelo solo puede llamar a `TIME_SERIES_DAILY`; el código procesa directamente su salida gratuita `compact`. Si el MCP no entrega la serie, consulta ese mismo endpoint por HTTPS con la llave de Alpha Vantage.

**Deberías ver:** `✅ Conexión preparada`.

In [ ]:
DESCARGO = "Nota educativa, no es recomendación de inversión."

class DatosAlphaVantageError(RuntimeError):
    """Fallo clasificable sin exponer la llave ni la respuesta completa."""
    def __init__(self, codigo):
        super().__init__(codigo)
        self.codigo = codigo
cliente = None
if not MODO_SIMULACION:
    from openai import OpenAI
    cliente = OpenAI(api_key=OPEN_API_KEY)

def llamar_openai(system, mensaje, usar_mcp=False, max_tokens=900):
    """Devuelve texto, herramientas MCP usadas y sus salidas originales."""
    base = dict(model=MODELO, max_output_tokens=max_tokens,
                instructions=system, input=mensaje)
    if usar_mcp:
        resp = cliente.responses.create(
            **base,
            tools=[{"type": "mcp", "server_label": "alphavantage",
                    "server_description": "Datos bursátiles para una actividad educativa.",
                    "server_url": "https://mcp.alphavantage.co/mcp",
                    "authorization": ALPHAVANTAGE_KEY,
                    "require_approval": "never",
                    "allowed_tools": ["TIME_SERIES_DAILY"]}])
    else:
        resp = cliente.responses.create(**base)
    texto = (resp.output_text or "").strip()
    herramientas = [item.name for item in resp.output
                     if item.type == "mcp_call" and getattr(item, "name", None)]
    salidas = [item.output for item in resp.output
               if item.type == "mcp_call" and getattr(item, "output", None)]
    return texto, herramientas, salidas

def extraer_json(texto):
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin <= inicio:
        raise ValueError("El agente no devolvió JSON")
    return json.loads(texto[inicio:fin + 1])

def a_numero(valor):
    try:
        return float(str(valor).replace("%", "").replace(",", ""))
    except (TypeError, ValueError):
        return None

def buscar_serie_diaria(objeto):
    """Localiza la serie diaria aunque el MCP la envuelva en bloques de contenido."""
    if isinstance(objeto, str):
        texto = objeto.strip().strip("`")
        try:
            return buscar_serie_diaria(json.loads(texto))
        except (json.JSONDecodeError, TypeError):
            inicio, fin = texto.find("{"), texto.rfind("}")
            if inicio != -1 and fin > inicio:
                try:
                    return buscar_serie_diaria(json.loads(texto[inicio:fin + 1]))
                except json.JSONDecodeError:
                    return None
        return None
    if isinstance(objeto, list):
        for elemento in objeto:
            encontrado = buscar_serie_diaria(elemento)
            if encontrado:
                return encontrado
        return None
    if not isinstance(objeto, dict):
        return None
    for clave, valor in objeto.items():
        if "time series" in str(clave).lower() and "daily" in str(clave).lower() and isinstance(valor, dict):
            return valor
    fechas = [clave for clave in objeto if re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(clave))]
    if fechas and isinstance(objeto[fechas[0]], dict):
        return objeto
    for valor in objeto.values():
        encontrado = buscar_serie_diaria(valor)
        if encontrado:
            return encontrado
    return None

def campo_alpha(fila, nombre):
    for clave, valor in fila.items():
        if str(clave).lower().strip().endswith(nombre):
            return a_numero(valor)
    return None

def rsi_wilder(cierres, periodo=14):
    if len(cierres) <= periodo:
        return None
    cambios = [actual - anterior for anterior, actual in zip(cierres, cierres[1:])]
    ganancias = [max(cambio, 0) for cambio in cambios]
    perdidas = [max(-cambio, 0) for cambio in cambios]
    promedio_ganancia = sum(ganancias[:periodo]) / periodo
    promedio_perdida = sum(perdidas[:periodo]) / periodo
    for ganancia, perdida in zip(ganancias[periodo:], perdidas[periodo:]):
        promedio_ganancia = (promedio_ganancia * (periodo - 1) + ganancia) / periodo
        promedio_perdida = (promedio_perdida * (periodo - 1) + perdida) / periodo
    if promedio_perdida == 0:
        return 100.0
    fuerza = promedio_ganancia / promedio_perdida
    return round(100 - 100 / (1 + fuerza), 1)

def ficha_desde_serie_diaria(ticker, serie):
    """Convierte OHLCV diario gratuito en una ficha reproducible."""
    filas = []
    for fecha in sorted(serie):
        fila = serie[fecha]
        if not isinstance(fila, dict):
            continue
        datos = {"fecha": fecha, "apertura": campo_alpha(fila, "open"),
                 "maximo": campo_alpha(fila, "high"), "minimo": campo_alpha(fila, "low"),
                 "cierre": campo_alpha(fila, "close"), "volumen": campo_alpha(fila, "volume")}
        if None not in (datos["maximo"], datos["minimo"], datos["cierre"]):
            filas.append(datos)
    if len(filas) < 2:
        raise ValueError("Alpha Vantage no devolvió suficientes sesiones diarias")

    cierres = [fila["cierre"] for fila in filas]
    retornos = [actual / anterior - 1 for anterior, actual in zip(cierres, cierres[1:]) if anterior]
    ultima, anterior = filas[-1], filas[-2]
    volumenes_20 = [fila["volumen"] for fila in filas[-20:] if fila["volumen"] is not None]
    promedio = lambda valores: round(sum(valores) / len(valores), 2) if valores else None
    volatilidad = (round(statistics.stdev(retornos[-20:]) * math.sqrt(252) * 100, 1)
                   if len(retornos) >= 20 else None)
    return {
        "ticker": ticker, "nombre": None, "sector": None,
        "precio": round(ultima["cierre"], 2),
        "tipo_precio": "cierre diario histórico; no es cotización en tiempo real",
        "cambio_pct": round((ultima["cierre"] / anterior["cierre"] - 1) * 100, 2),
        "fecha_dato": ultima["fecha"], "apertura": ultima["apertura"],
        "maximo_dia": ultima["maximo"], "minimo_dia": ultima["minimo"],
        "volumen": round(ultima["volumen"]) if ultima["volumen"] is not None else None,
        "promedio_volumen_20d": round(promedio(volumenes_20)) if volumenes_20 else None,
        "max_100_sesiones": round(max(f["maximo"] for f in filas[-100:]), 2),
        "min_100_sesiones": round(min(f["minimo"] for f in filas[-100:]), 2),
        "sma_20": promedio(cierres[-20:]) if len(cierres) >= 20 else None,
        "sma_50": promedio(cierres[-50:]) if len(cierres) >= 50 else None,
        "rsi_14": rsi_wilder(cierres),
        "retorno_20d_pct": round((cierres[-1] / cierres[-21] - 1) * 100, 2) if len(cierres) >= 21 else None,
        "volatilidad_20d_anual_pct": volatilidad, "sesiones_disponibles": len(filas),
        "fuente": "Alpha Vantage TIME_SERIES_DAILY compact vía MCP (datos gratuitos)"}


def consultar_serie_diaria_rest(ticker):
    """Respaldo directo al endpoint gratuito cuando el MCP no entrega la serie."""
    respuesta = requests.get(
        "https://www.alphavantage.co/query",
        params={"function": "TIME_SERIES_DAILY", "symbol": ticker,
                "outputsize": "compact", "apikey": ALPHAVANTAGE_KEY},
        timeout=20)
    respuesta.raise_for_status()
    datos = respuesta.json()
    serie = buscar_serie_diaria(datos)
    if not serie:
        mensajes = " ".join(str(datos.get(k, "")) for k in
                            ("Information", "Note", "Error Message") if isinstance(datos, dict)).lower()
        if "rate limit" in mensajes or "25 requests" in mensajes or "call frequency" in mensajes:
            codigo = "cuota_diaria"
        elif "invalid api key" in mensajes or "demo api key" in mensajes:
            codigo = "llave_no_valida"
        elif "premium" in mensajes:
            codigo = "acceso_premium"
        else:
            codigo = "respuesta_sin_serie"
        raise DatosAlphaVantageError(codigo)
    return serie

print("✅ Conexión preparada")

## Celda 4 · 🛰️ Iris, la Recolectora (Agente 1)
Curiosa y meticulosa: solicita `TIME_SERIES_DAILY` con `outputsize=compact`. El código convierte hasta 100 sesiones de apertura, máximo, mínimo, cierre y volumen en una ficha verificable. Si el MCP no entrega la serie, intenta el mismo endpoint gratuito por HTTPS; en modo real se detiene si ambos fallan y nunca presenta datos DEMO como reales.

**Deberías ver:** `✅ Iris lista`.

In [ ]:
PROMPT_RECOLECTOR = """Eres Iris, la Recolectora (Agente 1) de una mesa financiera educativa.
Tu personalidad: curiosa y meticulosa; jamás inventas un dato.
Tarea: obtener la serie diaria histórica de la acción indicada con Alpha Vantage.
Reglas:
- Haz exactamente 1 consulta con TIME_SERIES_DAILY y outputsize compact.
- No uses funciones premium ni afirmes que el dato es en tiempo real.
- El código leerá directamente la salida de la herramienta y hará todos los cálculos.
- Después de consultar responde solamente: {"estado":"consultado"}."""

DATOS_DEMO = {
    "nombre": "Empresa ficticia (simulación)", "sector": "Tecnología", "precio": 150.0,
    "cambio_pct": 1.2, "fecha_dato": "dato ficticio", "max_100_sesiones": 180.0,
    "min_100_sesiones": 120.0, "sma_20": 148.0, "sma_50": 145.0, "rsi_14": 55.0,
    "retorno_20d_pct": 2.4, "volatilidad_20d_anual_pct": 24.0, "volumen": 1000000,
    "promedio_volumen_20d": 900000, "sesiones_disponibles": 100,
    "tipo_precio": "dato ficticio de cierre diario"}

def agente_recolector(ticker):
    if MODO_SIMULACION:
        return {"ticker": ticker, **DATOS_DEMO,
                "fuente": "DATOS DEMO, no son reales (modo simulación)"}, []
    herramientas = []
    try:
        _, herramientas, salidas = llamar_openai(
            PROMPT_RECOLECTOR, f"Acción: {ticker}", usar_mcp=True, max_tokens=200)
        serie = buscar_serie_diaria(salidas)
        if not serie:
            raise ValueError("TIME_SERIES_DAILY no devolvió una serie utilizable")
        ficha = ficha_desde_serie_diaria(ticker, serie)
        ficha["canal_datos"] = "MCP"
        return ficha, herramientas
    except Exception as error_mcp:
        try:
            ficha = ficha_desde_serie_diaria(ticker, consultar_serie_diaria_rest(ticker))
            ficha["fuente"] = ("Alpha Vantage TIME_SERIES_DAILY compact vía REST "
                               "(datos gratuitos; respaldo del MCP)")
            ficha["canal_datos"] = "REST"
            return ficha, herramientas
        except Exception as error_rest:
            codigo = getattr(error_rest, "codigo", "conexion_o_respuesta")
            raise DatosAlphaVantageError(codigo) from error_mcp

print("✅ Iris lista")

## Celda 5 · 🧮 Euler, el Analista (Agente 2)
**Regla de oro:** los números los calcula el código; la IA solo interpreta.

Con los datos diarios gratuitos calcula: posición dentro del rango de hasta 100 sesiones, SMA 20/50, RSI 14, retorno de 20 sesiones, volatilidad histórica anualizada y volumen relativo al promedio de 20 sesiones. Son señales descriptivas, no predicciones ni recomendaciones.

**Deberías ver:** un diccionario con esos indicadores y `✅ Euler listo`.

In [ ]:
def calcular_indicadores(ficha):
    precio, maximo, minimo = (a_numero(ficha.get(k)) for k in
                              ("precio", "max_100_sesiones", "min_100_sesiones"))
    posicion = (round((precio - minimo) / (maximo - minimo) * 100, 1)
                if None not in (precio, maximo, minimo) and maximo > minimo else None)
    sma_20, sma_50 = a_numero(ficha.get("sma_20")), a_numero(ficha.get("sma_50"))
    if None not in (precio, sma_20, sma_50) and precio > sma_20 > sma_50:
        tendencia = "🟢 cierre sobre SMA 20 y SMA 50"
    elif None not in (precio, sma_20, sma_50) and precio < sma_20 < sma_50:
        tendencia = "🔴 cierre bajo SMA 20 y SMA 50"
    else:
        tendencia = "🟡 tendencia mixta o sin datos suficientes"
    rsi = a_numero(ficha.get("rsi_14"))
    if rsi is None: zona_rsi = "⚪ RSI no disponible"
    elif rsi > 70: zona_rsi = "🟠 RSI por encima de 70; posible sobrecompra, no señal automática"
    elif rsi < 30: zona_rsi = "🔵 RSI por debajo de 30; posible sobreventa, no señal automática"
    else: zona_rsi = "🟢 RSI entre 30 y 70"
    volatilidad = a_numero(ficha.get("volatilidad_20d_anual_pct"))
    if volatilidad is None: riesgo = "⚪ volatilidad no disponible"
    elif volatilidad < 20: riesgo = "🟢 volatilidad histórica baja"
    elif volatilidad <= 40: riesgo = "🟡 volatilidad histórica moderada"
    else: riesgo = "🔴 volatilidad histórica alta"
    volumen, promedio = a_numero(ficha.get("volumen")), a_numero(ficha.get("promedio_volumen_20d"))
    volumen_relativo = round(volumen / promedio, 2) if None not in (volumen, promedio) and promedio > 0 else None
    return {"posicion_rango_100_sesiones_pct": posicion, "tendencia_medias": tendencia,
            "zona_rsi_14": zona_rsi, "rsi_14": rsi,
            "retorno_20d_pct": a_numero(ficha.get("retorno_20d_pct")),
            "volatilidad_20d_anual_pct": volatilidad, "semaforo_volatilidad": riesgo,
            "volumen_relativo_20d": volumen_relativo, "cambio_pct": a_numero(ficha.get("cambio_pct"))}

PROMPT_ANALISTA = """Eres Euler, el Analista (Agente 2). Recibes la ficha de Iris y los indicadores calculados.
Tu personalidad: escéptico constructivo, riguroso con los números y paciente para explicarlos.
NO recalcules ni cambies cifras. El precio es el último cierre diario histórico, no una cotización en tiempo real.
No hay P/E ni beta porque esas funciones no forman parte del conjunto gratuito utilizado.
Responde SOLO con JSON con las claves: lectura_precio, lectura_tendencia, lectura_riesgo,
lectura_volumen y pregunta_para_investigar. Cada lectura debe ser una frase breve."""

def agente_analista(ficha):
    ind = calcular_indicadores(ficha)
    if MODO_SIMULACION:
        lectura = {
            "lectura_precio": f"El cierre está al {ind['posicion_rango_100_sesiones_pct']} % del rango observado.",
            "lectura_tendencia": ind["tendencia_medias"],
            "lectura_riesgo": ind["semaforo_volatilidad"],
            "lectura_volumen": f"Volumen relativo: {ind['volumen_relativo_20d']} veces el promedio de 20 sesiones.",
            "pregunta_para_investigar": "¿Qué noticia podría confirmar o contradecir este patrón?"}
    else:
        texto, _, _ = llamar_openai(PROMPT_ANALISTA,
                                    json.dumps({"ficha": ficha, "indicadores": ind}, ensure_ascii=False))
        try:
            lectura = extraer_json(texto)
        except ValueError:
            lectura = {"lectura_precio": texto[:300]}
    return {"ficha": ficha, "indicadores": ind, "lectura": lectura}

print(calcular_indicadores(DATOS_DEMO))  # comprobación reproducible
print("✅ Euler listo")

## Celda 6 · ✍️ Vera, la Comunicadora (Agente 3)
Clara, breve y honesta. **Deberías ver:** `✅ Vera lista`.

In [ ]:
PROMPT_COMUNICADOR = f"""Eres Vera, la Comunicadora (Agente 3). Redacta una nota para Telegram.
Tu personalidad: clara, breve y honesta sobre los límites del análisis.
- Máximo 900 caracteres, español claro, 3 a 5 emojis como máximo.
- Estructura: ticker y fecha · último cierre y cambio · SMA 20/50 y RSI 14 · volatilidad y volumen relativo.
- Aclara que es cierre diario histórico, no una cotización en tiempo real.
- Usa solo las cifras recibidas; si la fuente dice DEMO, adviértelo en la primera línea.
- Cierra SIEMPRE con: "{DESCARGO}"
Texto plano, sin Markdown."""

def agente_comunicador(analisis):
    f, ind, lec = analisis["ficha"], analisis["indicadores"], analisis["lectura"]
    if MODO_SIMULACION:
        return (f"✍️ Vera · Comunicadora\n⚠️ {f['fuente']}\n📊 {f['ticker']}\n"
                f"Cierre: {f['precio']} USD ({ind['cambio_pct']} % frente a la sesión previa)\n"
                f"Tendencia: {ind['tendencia_medias']} · RSI {ind['rsi_14']}\n"
                f"Riesgo: {ind['semaforo_volatilidad']}\n{lec['lectura_volumen']}\n"
                f"🔎 {lec['pregunta_para_investigar']}\n\n{DESCARGO}")
    texto, _, _ = llamar_openai(PROMPT_COMUNICADOR, json.dumps(analisis, ensure_ascii=False), max_tokens=600)
    texto = texto if DESCARGO in texto else f"{texto}\n\n{DESCARGO}"
    return f"✍️ Vera · Comunicadora\n{texto}"

print("✅ Vera lista")

## Celda 7 · 🎼 Batuta, el orquestador, y una prueba sin Telegram
Batuta decide el orden y **pasa el output de cada agente como input del siguiente**.

**Deberías ver:** las entregas `1→2` y `2→3` y, al final, la nota con el descargo. En modo real, la ficha debe identificar `TIME_SERIES_DAILY compact`, incluir una fecha y mostrar indicadores calculados. Toma **captura** de esta salida.

In [ ]:
def pipeline(ticker, avisar=print):
    ticker = ticker.strip().upper()
    if not re.fullmatch(r"[A-Z0-9.\-]{1,12}", ticker):
        raise ValueError(f"Ticker no válido: {ticker}")
    avisar(f"🎼 Batuta · Orquestador\nRecibí tu orden. Inicio la cadena para {ticker} (modelo {MODELO}).")
    avisar(f"🛰️ Iris · Recolectora buscando datos de {ticker}…")
    ficha, herramientas = agente_recolector(ticker)
    if MODO_CLASE:
        avisar("📦 Entrega 1→2 · Iris → Euler (ficha):\n" + json.dumps(ficha, ensure_ascii=False, indent=1)[:1500]
               + f"\n🔧 Herramientas MCP usadas: {', '.join(herramientas) or 'ninguna'}")
    avisar("🧮 Euler · Analista calculando e interpretando…")
    analisis = agente_analista(ficha)
    if MODO_CLASE:
        avisar("📦 Entrega 2→3 · Euler → Vera (indicadores):\n" + json.dumps(analisis["indicadores"], ensure_ascii=False, indent=1))
    avisar("✍️ Vera · Comunicadora redactando la nota…")
    return agente_comunicador(analisis)

print(pipeline("IBM"))

## Celda 8 · 📱 Telegram como centro de mando
Al ejecutar esta celda el bot queda **escuchando**. Escríbele desde Telegram: `/analiza AAPL`.

- Para apagarlo, pulsa ■ (detener) en la celda.
- Si cierras la pestaña, el bot se apaga. Para que funcione con la computadora apagada usa la **Ruta B (GitHub Actions)**.

**Deberías ver:** `✅ Bot @tu_bot escuchando` y, en Telegram, los mensajes de Batuta, Iris, Euler y Vera.

In [ ]:
API_TG = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
AYUDA = ("🤖 Mesa Financiera de 3 agentes\n/analiza TICKER → ejemplo: /analiza AAPL\n"
         "Batuta dirige: Iris (MCP) → Euler → Vera")

def solicitar_tg(metodo, verbo="get", timeout=20, **kwargs):
    try:
        peticion = requests.post if verbo == "post" else requests.get
        respuesta = peticion(f"{API_TG}/{metodo}", timeout=timeout, **kwargs)
        respuesta.raise_for_status()
        datos = respuesta.json()
    except (requests.RequestException, ValueError):
        raise RuntimeError(f"Telegram no respondió correctamente ({metodo}).") from None
    if not isinstance(datos, dict) or not datos.get("ok"):
        raise RuntimeError(f"Telegram rechazó {metodo}.")
    return datos

def enviar(chat_id, texto):
    return solicitar_tg("sendMessage", verbo="post",
                        data={"chat_id": chat_id, "text": texto[:4000]})

def atender(mensaje):
    chat_id, texto = mensaje["chat"]["id"], (mensaje.get("text") or "").strip()
    if not TELEGRAM_CHAT_ID or str(chat_id) != TELEGRAM_CHAT_ID:
        return None
    if not texto.lower().startswith("/analiza"):
        return enviar(chat_id, AYUDA)
    partes = texto.split()
    try:
        nota = pipeline(partes[1] if len(partes) > 1 else "", avisar=lambda t: enviar(chat_id, t))
        enviar(chat_id, nota)
    except ValueError:
        enviar(chat_id, "Escribe un ticker válido. Ejemplo: /analiza MSFT")
    except Exception as e:
        enviar(chat_id, f"⚠️ No pude completar el análisis ({type(e).__name__}). Revisa llaves y saldo.")

def encender_bot():
    if not TELEGRAM_CHAT_ID:
        raise RuntimeError("🔒 Falta TELEGRAM_CHAT_ID: el bot se bloqueó para impedir acceso público.")
    yo = solicitar_tg("getMe")
    solicitar_tg("deleteWebhook")
    print(f"✅ Bot @{yo['result']['username']} escuchando. Pulsa ■ para apagarlo.")
    offset = None
    try:
        while True:
            try:
                r = solicitar_tg("getUpdates", params={"timeout": 30, "offset": offset}, timeout=45)
                for upd in r.get("result", []):
                    offset = upd["update_id"] + 1
                    if "message" in upd:
                        atender(upd["message"])
            except (requests.RequestException, RuntimeError):
                print("Conexión inestable con Telegram; reintento en 5 s.")
                time.sleep(5)
    except KeyboardInterrupt:
        print("👋 Bot detenido.")

if TELEGRAM_TOKEN:
    encender_bot()
else:
    print("⬜ Falta TELEGRAM_BOT_TOKEN: crea tu bot con @BotFather y guárdalo en 🔑 Secretos.")

## ✅ Lista de validación
- [ ] La celda 7 muestra las entregas 1→2 y 2→3.
- [ ] En modo real, la fuente dice **TIME_SERIES_DAILY compact**, aparece esa herramienta y hay fecha del dato.
- [ ] La nota aclara que el valor es un cierre diario histórico y no tiempo real.
- [ ] En Telegram, `/analiza AAPL` devuelve tres avisos y una nota final con el descargo.
- [ ] `/analiza AAPL;x` responde "Escribe un ticker válido".

> Material educativo de la Dra. Elda C. Morales. Las notas **no son recomendaciones de inversión**.